# all-reduce-eval-metrics — faded example 3: Issue the all_reduce that syncs the per-class count matrix

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-eval-metrics`. The last cell reports your progress on the `Distributed: all_reduce eval metrics` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce eval metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-eval-metrics`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-eval-metrics"
DD_SUBTOPIC = "Distributed: all_reduce eval metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Per-class correct and total counts are stacked into a `2 x C` tensor; one `all_reduce(SUM)` sums every entry across ranks in place, after which an element-wise divide gives per-class accuracy. The blanked step is the reduce itself — the in-place collective that turns each rank's local counts into the global counts shared by all ranks.

## Faded exercise 3

The packed `2 x C` tensor and the per-class divide are written. The learner must issue the single in-place collective that sums the packed tensor across all ranks (using SUM). Fill in the blanked function body line that performs the all_reduce.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def per_class_accuracy(dist_module, local_correct, local_total):
    packed = t.stack([local_correct.float(), local_total.float()])  # (2, C)
    dist_module.all_reduce(packed, op=dist_module.ReduceOp.SUM)
    return packed[0] / packed[1]

t.manual_seed(0)
r0_correct, r0_total = t.tensor([5, 3, 4]), t.tensor([6, 4, 5])
r1_correct, r1_total = t.tensor([2, 7, 1]), t.tensor([3, 9, 2])
rank_tensors = [
    t.stack([r0_correct.float(), r0_total.float()]),
    t.stack([r1_correct.float(), r1_total.float()]),
]
mock = MockDist(rank_tensors)
per_class = per_class_accuracy(mock, r0_correct, r0_total)
print('per-class accuracy:', [round(x, 4) for x in per_class.tolist()])


def _test():
    r0_correct, r0_total = t.tensor([5, 3, 4]), t.tensor([6, 4, 5])
    r1_correct, r1_total = t.tensor([2, 7, 1]), t.tensor([3, 9, 2])
    global_correct = (r0_correct + r1_correct).float()
    global_total = (r0_total + r1_total).float()
    expected = global_correct / global_total
    rank_tensors = [
        t.stack([r0_correct.float(), r0_total.float()]),
        t.stack([r1_correct.float(), r1_total.float()]),
    ]
    mock = MockDist(rank_tensors)
    got = per_class_accuracy(mock, r0_correct, r0_total)
    assert got.shape == (3,), f'expected shape (3,), got {tuple(got.shape)}'
    assert t.allclose(got, expected, atol=1e-6), f'got {got.tolist()}, expected {expected.tolist()}'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def per_class_accuracy(dist_module, local_correct, local_total):
    packed = t.stack([local_correct.float(), local_total.float()])  # (2, C)
    dist_module.all_reduce(packed, op=dist_module.ReduceOp.SUM)
    return packed[0] / packed[1]

t.manual_seed(0)
r0_correct, r0_total = t.tensor([5, 3, 4]), t.tensor([6, 4, 5])
r1_correct, r1_total = t.tensor([2, 7, 1]), t.tensor([3, 9, 2])
rank_tensors = [
    t.stack([r0_correct.float(), r0_total.float()]),
    t.stack([r1_correct.float(), r1_total.float()]),
]
mock = MockDist(rank_tensors)
per_class = per_class_accuracy(mock, r0_correct, r0_total)
print('per-class accuracy:', [round(x, 4) for x in per_class.tolist()])
```
</details>